In [1]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable


In [3]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'MUCollective/multidy',
    'ASKurz/Doing-Bayesian-Data-Analysis-in-brms-and-the-tidyverse',
    'ESMValGroup/ESMValTool',
    'RTIInternational/SMART',
    'jannisborn/covid_detector',
    'OverLordGoldDragon/ssqueezepy',
    'rust-bio/rust-bio',
    'ccsb-scripps/AutoDock-GPU',
    'EBIvariation/eva-tools',
    'opendatacube/datacube-alchemist'
]

list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,005dd50bc16720a653f2cee00e786c551a1bf093,mucollective_multidy


In [6]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 40)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  7%|█████▍                                                                          | 90/1329 [01:32<21:31,  1.04s/it]

Got Errors {'0c7cf7ff6bba237c94042cdd79692e37bb5f8e15': 'Key 0c7cf7ff6bba237c94042cdd79692e37bb5f8e15 not found in /da5_fast/All.sha1c/commit_12.tch'}


 14%|███████████▏                                                                   | 189/1329 [03:16<19:51,  1.05s/it]

Got Errors {'24b8c0ab6ccc0ad691910148f50872a826843afa': 'Key 24b8c0ab6ccc0ad691910148f50872a826843afa not found in /da5_fast/All.sha1c/commit_36.tch'}


 38%|█████████████████████████████▋                                                 | 499/1329 [08:41<14:24,  1.04s/it]

...

Got Errors {'f004a368e767c9054e2febe6b054562e11bf454c': 'Key f004a368e767c9054e2febe6b054562e11bf454c not found in /da5_fast/All.sha1c/commit_112.tch'}


 93%|████████████████████████████████████████████████████████████████████████▎     | 1233/1329 [21:26<01:39,  1.04s/it]

Got Errors {'fd8fe18d20c92997024f5b588ae8235bf5f8be55': 'Key fd8fe18d20c92997024f5b588ae8235bf5f8be55 not found in /da5_fast/All.sha1c/commit_125.tch', 'fdafcc1693f2dd084759f6a4d45ac8328571a710': 'Key fdafcc1693f2dd084759f6a4d45ac8328571a710 not found in /da5_fast/All.sha1c/commit_125.tch', 'fdb3c25c14f6494cea6e64c15aeb3a0dbdeb1d45': 'Key fdb3c25c14f6494cea6e64c15aeb3a0dbdeb1d45 not found in /da5_fast/All.sha1c/commit_125.tch', 'fdf06c9234dff2314ab8de0bce4b77438f5c6c8f': 'Key fdf06c9234dff2314ab8de0bce4b77438f5c6c8f not found in /da5_fast/All.sha1c/commit_125.tch'}


100%|██████████████████████████████████████████████████████████████████████████████| 1329/1329 [23:06<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,005dd50bc16720a653f2cee00e786c551a1bf093,8f645b67eb916ec45bf73d6f3133c655b0a0c01f,[5368afd933c40fb2505dc768ac4bc3b12e4953dd],Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1603765446,-0500,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1603765446,-0500,solves #54\n,005dd50bc16720a653f2cee00e786c551a1bf093,mucollective_multidy
1,0075221fdb8bb0e15b9a4aa25984cc14ad95f93a,32c9b0fcf053d6b4913586cb4e0a4b39e54c4a73,[ceaebe2fd1c779a4802e29ee2e5b4bdb89babe37],Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1645114955,-0600,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1645114955,-0600,updated docs\n,0075221fdb8bb0e15b9a4aa25984cc14ad95f93a,mucollective_multidy


In [7]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '005dd50bc16720a653f2cee00e786c551a1bf093', 'tree': '8f645b67eb916ec45bf73d6f3133c655b0a0c01f', 'parent': ['5368afd933c40fb2505dc768ac4bc3b12e4953dd'], 'author': 'Abhraneel Sarma <abhraneelsarma2024@u.northwestern.edu>', 'author_time': 1603765446, 'author_tz': '-0500', 'committer': 'Abhraneel Sarma <abhraneelsarma2024@u.northwestern.edu>', 'committer_time': 1603765446, 'committer_tz': '-0500', 'message': 'solves #54\n'}


In [13]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = df_commit_data[
    ["project", "commit", "author", "author_time", "message"]
].copy()

# Rename columns to match the assignment requirements
dfinf.columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

# Check if it has the right content
dfinf.head()


,project_wocid,commit_sha1,author,time,commit message
0,mucollective_multidy,005dd50bc16720a653f2cee00e786c551a1bf093,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1603765446,solves #54\n
1,mucollective_multidy,0075221fdb8bb0e15b9a4aa25984cc14ad95f93a,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1645114955,updated docs\n
2,mucollective_multidy,00773e3985875d42b894577caf71f722f3abfb65,Abhraneel Sarma <abhsarma@umich.edu>,1562946264,Merge branch 'dev' into no-ds-store
3,mucollective_multidy,00b1ff5cfa60395631845e4a1a28dad3dd8ac740,Abhraneel Sarma <abhraneelsarma2024@u.northwe...,1596691697,block exec logic fix\n
4,mucollective_multidy,00c48a03d1631fa015d4889f9a2b3e23fa5bb89e,Abhraneel Sarma <abhraneel@Abhraneels-MacBook-...,1620663885,eval fixes\n


In [14]:
# check if it has the right content
dfinf.head(1)

,project_wocid,commit_sha1,author,time,commit message
0,mucollective_multidy,005dd50bc16720a653f2cee00e786c551a1bf093,Abhraneel Sarma <abhraneelsarma2024@u.northwes...,1603765446,solves #54\n


In [15]:
#mode a means append, so you have all your projects in the same file
yournetid='pvathana'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

In [16]:
print("Number of rows:", len(dfinf))
print("Number of projects:", dfinf["project_wocid"].nunique())

dfinf.groupby("project_wocid").size()

Number of rows: 13268
Number of projects: 10


project_wocid
askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse      258
ccsb-scripps_autodock-gpu                                          370
ebivariation_eva-tools                                             310
esmvalgroup_esmvaltool                                           10353
jannisborn_covid_detector                                          140
mucollective_multidy                                               120
opendatacube_datacube-alchemist                                    280
overlordgolddragon_ssqueezepy                                      127
rtiinternational_smart                                             540
rust-bio_rust-bio                                                  770
dtype: int64

In [17]:
project_stats = df_commit_data.groupby("project").agg(
    number_of_commits=("commit", "nunique"),
    number_of_authors=("author", "nunique"),
    min_time=("author_time", "min"),
    max_time=("author_time", "max")
)

project_stats["min_time"] = pd.to_datetime(
    project_stats["min_time"], unit="s"
)

project_stats["max_time"] = pd.to_datetime(
    project_stats["max_time"], unit="s"
)

project_stats

,number_of_commits,number_of_authors,min_time,max_time
project,,,,
askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse,258,7,2018-07-30 23:54:19,2025-04-15 14:27:35
ccsb-scripps_autodock-gpu,370,25,2017-06-13 17:36:32,2025-01-30 22:41:15
ebivariation_eva-tools,310,29,2015-11-02 13:59:17,2025-05-04 06:51:23
esmvalgroup_esmvaltool,10353,295,2017-01-26 14:15:37,2026-04-17 11:04:12
jannisborn_covid_detector,140,18,2020-03-28 15:17:49,2024-07-09 17:56:57
mucollective_multidy,120,15,2019-06-10 14:19:20,2025-10-08 20:48:42
opendatacube_datacube-alchemist,280,25,2019-08-07 06:31:12,2025-11-03 17:06:22
overlordgolddragon_ssqueezepy,127,6,2020-06-12 11:31:59,2025-08-02 02:30:27
rtiinternational_smart,540,33,2017-05-18 17:39:52,2025-02-01 06:42:54


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [ ]:
# World of Code Project Statistics

## The table above shows the number of commits, number of unique authors, minimum commit time, and maximum commit time retrieved from World of Code for each assigned project.

In [ ]:
## GitHub Repository Information

| Project | Stars | Forks | Last Commit Date |

| MUCollective/multidy |67 stars |5 forks|Sep 11, 2026 |
| ASKurz/Doing-Bayesian-Data-Analysis-in-brms-and-the-tidyverse |154 stars |43 forks |Jan 4, 2026 |
| ESMValGroup/ESMValTool |276 stars |153 forks |Sep 22, 2026|
| RTIInternational/SMART |232 stars |31 forks |Dec 2, 2024 |
| jannisborn/covid_detector |174 stars |81 forks |Apr 18, 2025 |
| OverLordGoldDragon/ssqueezepy |805 stars |106 forks |Aug 23, 2025 |
| rust-bio/rust-bio |1.8k stars |221 forks |Sep 16, 2026 |
| ccsb-scripps/AutoDock-GPU |615 stars |151 forks |Aug 11, 2026 |
| EBIvariation/eva-tools |1 star |13 forks |Jul 24, 2026 |
| opendatacube/datacube-alchemist |22 stars |6 forks |Sep 28, 2026 |

In [ ]:
## World of Code Project Statistics

askurz_doing-bayesian-data-analysis-in-brms-and-the-tidyverse: 258 commits, 7 authors, minimum time: 2018-07-30 23:54:19, maximum time: 2025-04-15 14:27:35
ccsb-scripps_autodock-gpu: 370 commits, 25 authors, minimum time: 2017-06-13 17:36:32, maximum time: 2025-01-30 22:41:15
ebivariation_eva-tools: 310 commits, 29 authors, minimum time: 2015-11-02 13:59:17, maximum time: 2025-05-04 06:51:23
esmvalgroup_esmvaltool: 10,353 commits, 295 authors, minimum time: 2017-01-26 14:15:37, maximum time: 2026-04-17 11:04:12
jannisborn_covid_detector: 140 commits, 18 authors, minimum time: 2020-03-28 15:17:49, maximum time: 2024-07-09 17:56:57
mucollective_multidy: 120 commits, 15 authors, minimum time: 2019-06-10 14:19:20, maximum time: 2025-10-08 20:48:42
opendatacube_datacube-alchemist: 280 commits, 25 authors, minimum time: 2019-08-07 06:31:12, maximum time: 2025-11-03 17:06:22
overlordgolddragon_ssqueezepy: 127 commits, 6 authors, minimum time: 2020-06-12 11:31:59, maximum time: 2025-08-02 02:30:27
rtiinternational_smart: 540 commits, 33 authors, minimum time: 2017-05-18 17:39:52, maximum time: 2025-02-01 06:42:54
rust-bio_rust-bio: 770 commits, 120 authors, minimum time: 2015-01-16 16:43:41, maximum time: 2025-11-03 04:04:38